# 从零到项目上线

本章以 `Zesay` 为例，搭建一个前后端分离的全栈项目。

- 前端二选一：**Next.js** 或 **React + Vite**
- 后端：**FastAPI + uv**
- 版本管理：**Git**，只在项目根目录初始化一次

> 推荐初学者优先选择 Next.js。若课程明确要求 React + Vite，则使用 Vite 路线。不要在同一个 `frontend/` 中同时初始化两套框架。

## 1. 准备环境

安装 Node.js、Python、uv 和 Git，然后检查版本：

```powershell
node --version
npm --version
python --version
uv --version
git --version
```

## 2. 创建项目根目录并初始化 Git

```powershell
mkdir Zesay
cd Zesay
mkdir backend
mkdir docs
git init
```

Git 应在 `Zesay/` 根目录初始化，而不是放在 `frontend/` 或 `backend/` 中。这样一个仓库可以统一管理前端、后端和文档。除非明确要拆成多个仓库，否则不要在子目录再次执行 `git init`。

最终结构：

```text
Zesay/
├─ frontend/       # Next.js，或 React + Vite
├─ backend/        # FastAPI + uv
├─ docs/           # 需求、设计和接口文档
├─ README.md
├─ .gitignore
└─ AGENTS.md
```

## 3A. 前端方案一：Next.js（推荐）

在 `Zesay/` 根目录执行：

```powershell
npx create-next-app@latest frontend
```

交互选项建议：

```text
TypeScript                              Yes
ESLint                                 Yes
React Compiler                         Yes（如果出现）
Tailwind CSS                           Yes
代码放入 src/ 目录                     Yes
App Router                             Yes
Turbopack                              Yes（如果出现）
Customize the default import alias?    No
```

最后一项选择 `No`，保留默认的 `@/*` 导入别名。启动前端：

```powershell
cd frontend
npm run dev
```

默认地址通常是 <http://localhost:3000>。

## 3B. 前端方案二：React + Vite

如果不用 Next.js，在 `Zesay/` 根目录执行：

```powershell
npm create vite@latest frontend -- --template react-ts
cd frontend
npm install
npm run dev
```

默认地址通常是 <http://localhost:5173>。React 负责构建界面；Vite 负责项目创建、开发服务器、热更新和生产构建，因此 Vite 也是这套前端方案的一部分。

## 4. 创建 FastAPI + uv 后端

回到 `Zesay/` 根目录执行：

```powershell
cd backend
uv init
uv add fastapi "uvicorn[standard]"
```

`uv init` 只初始化 Python 项目；还必须安装 FastAPI 和 Uvicorn。三者分工如下：

- `uv`：管理 Python、虚拟环境和依赖
- `FastAPI`：编写 Web API
- `Uvicorn`：运行 FastAPI 应用

创建 `backend/main.py`：

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="Zesay API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "http://localhost:5173"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/api/health")
def health_check() -> dict[str, str]:
    return {"status": "ok"}
```

启动后端：

```powershell
uv run uvicorn main:app --reload
```

- 健康检查：<http://127.0.0.1:8000/api/health>
- Swagger 文档：<http://127.0.0.1:8000/docs>

## 5. 前后端联调

分别打开两个终端。

终端一：

```powershell
cd Zesay/backend
uv run uvicorn main:app --reload
```

终端二：

```powershell
cd Zesay/frontend
npm run dev
```

前端测试请求：

```ts
const response = await fetch("http://127.0.0.1:8000/api/health");
const data = await response.json();
console.log(data.status); // ok
```

## 6. 环境变量

Next.js 的 `frontend/.env.local`：

```dotenv
NEXT_PUBLIC_API_URL=http://127.0.0.1:8000
```

读取方式：`process.env.NEXT_PUBLIC_API_URL`。

Vite 的 `frontend/.env.local`：

```dotenv
VITE_API_URL=http://127.0.0.1:8000
```

读取方式：`import.meta.env.VITE_API_URL`。

> `NEXT_PUBLIC_` 和 `VITE_` 开头的变量会暴露给浏览器，不能存放 API Key、数据库密码等秘密。

## 7. 根目录基础文件

`.gitignore`：

```gitignore
# Frontend
frontend/node_modules/
frontend/.next/
frontend/dist/

# Backend
backend/.venv/
backend/__pycache__/
backend/.pytest_cache/
*.py[cod]

# Environment files
.env
.env.*
!.env.example

# IDE and OS
.vscode/
.idea/
.DS_Store
Thumbs.db
```

`README.md` 至少应说明项目用途、技术栈、安装步骤、启动命令、环境变量、测试和部署方式。

`AGENTS.md` 可约定目录职责、代码风格、测试要求、`/api/` 路径规范，以及禁止提交密钥等规则。

## 8. 检查并提交

前端检查：

```powershell
cd frontend
npm run lint
npm run build
```

后端增加测试后执行：

```powershell
cd backend
uv add --dev pytest
uv run pytest
```

全部通过后，在 `Zesay/` 根目录提交：

```powershell
git status
git add .
git commit -m "chore: initialize Zesay project"
```

## 9. 部署上线

常见组合：Next.js 部署到 Vercel 或 Node.js 平台；React + Vite 部署到 Vercel、Netlify 或静态托管；FastAPI 部署到支持 Python 或容器的平台；数据库使用托管 PostgreSQL。

上线步骤：

1. 将代码推送到远程 Git 仓库。
2. 分别创建前端和后端部署服务。
3. 在平台设置生产环境变量，不把密钥写进仓库。
4. 将前端 API 地址改为线上后端地址。
5. 将 FastAPI 的 CORS 白名单改为真实前端域名。
6. 执行构建、测试和健康检查。
7. 配置自定义域名、HTTPS、日志和监控。

## 总结

```text
前端：Next.js 或 React + Vite（二选一）
后端：uv 初始化 Python 项目，再安装 FastAPI 和 Uvicorn
仓库：Git 在 Zesay/ 根目录初始化一次，统一管理全部内容
```